# Built-in DSPy and HuggingFace Datasets

Load HotPotQA and GSM8K through DSPy's built-in dataset wrappers, then pull SQuAD and TweetEval from the HuggingFace Hub and convert each one into `dspy.Example` objects you can feed into an optimizer.

**Required env vars:** `OPENAI_API_KEY` (loaded from `.env`).

In [ ]:
%pip install -r ../requirements.txt -q
%pip install datasets -q

In [ ]:
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM('openai/gpt-5-mini'))

## HotPotQA — multi-hop question answering

Requires reasoning over multiple Wikipedia paragraphs to answer a question.

In [ ]:
from dspy.datasets import HotPotQA

# Load dataset
dataset = HotPotQA(train_seed=1, train_size=100, eval_seed=2024, dev_size=50, test_size=0)

# Convert to DSPy format
train_set = [dspy.Example(question=x.question, answer=x.answer).with_inputs('question')
            for x in dataset.train]

# Example
print(train_set[0].question)
print(train_set[0].answer)

## GSM8K — grade-school math word problems

Tests multi-step arithmetic reasoning.

In [ ]:
from dspy.datasets import GSM8K

dataset = GSM8K()
train_set = [dspy.Example(question=x.question, answer=x.answer).with_inputs('question')
            for x in dataset.train[:100]]

print(train_set[0].question)
print(train_set[0].answer)

## SQuAD via HuggingFace

`datasets.load_dataset` pulls the Stanford Question Answering Dataset. Convert each row into a `dspy.Example` with the question, context, and the first listed answer.

In [ ]:
from datasets import load_dataset

# Load from HuggingFace
hf_dataset = load_dataset("squad")

# Convert to DSPy format
def convert_to_dspy(hf_example):
    return dspy.Example(
        question=hf_example['question'],
        context=hf_example['context'],
        answer=hf_example['answers']['text'][0],  # SQuAD has multiple answers, take first
    ).with_inputs('question', 'context')

train_set = [convert_to_dspy(ex) for ex in hf_dataset['train'].select(range(1000))]

print(train_set[0].question)
print(train_set[0].answer)

## Optional: optimize a sample and evaluate on held-out data

Large public datasets can make optimization and evaluation expensive. This example follows the chapter's recommendation: deterministically sample 200 training examples, use 100 held-out validation examples while MIPROv2 searches, and reserve another 500 examples for final evaluation. Running the cell performs the complete optimization and evaluation workflow.

In [ ]:
import random

train_set_full = list(train_set)
train_set_sample = random.Random(2024).sample(train_set_full, 200)
val_set_sample = [
    convert_to_dspy(ex)
    for ex in hf_dataset['validation'].select(range(100))
]
test_set = [
    convert_to_dspy(ex)
    for ex in hf_dataset['validation'].select(range(100, 600))
]

class ExtractiveQA(dspy.Signature):
    """Answer a question using the supplied context."""
    question: str = dspy.InputField()
    context: str = dspy.InputField()
    answer: str = dspy.OutputField()

def exact_match(example, pred, trace=None):
    return example.answer.strip().lower() == pred.answer.strip().lower()

program = dspy.ChainOfThought(ExtractiveQA)
optimizer = dspy.MIPROv2(metric=exact_match, auto="light")
optimized_program = optimizer.compile(
    student=program,
    trainset=train_set_sample,
    valset=val_set_sample,
)

evaluator = dspy.Evaluate(
    devset=test_set,
    metric=exact_match,
    display_progress=True,
)
result = evaluator(optimized_program)
print(f"Held-out accuracy: {result.score:.1f}%")

## TweetEval sentiment — a classification dataset

Map the integer labels back to readable strings before wrapping each row as a `dspy.Example`.

In [ ]:
# Twitter sentiment analysis
hf_dataset = load_dataset("tweet_eval", "sentiment")

def convert_to_dspy(hf_example):
    label_map = {0: "negative", 1: "neutral", 2: "positive"}
    return dspy.Example(
        text=hf_example['text'],
        sentiment=label_map[hf_example['label']]
    ).with_inputs('text')

train_set = [convert_to_dspy(ex) for ex in hf_dataset['train']]

print(f"Loaded {len(train_set)} sentiment examples")
print(train_set[0].text, '->', train_set[0].sentiment)